# Vocational NMT: run 7, five more languages (Marathi, Gujarati, Tamil, Telugu, Kannada)

1. **Teacher drafts + terminology repair** for the 5 new languages. GPU 0: mar_Deva, tam_Taml, guj_Gujr; GPU 1: tel_Telu, kan_Knda. The original 4 languages' drafts ship in the bundle and aren't re-translated.
2. **Training:** GPU 0 trains `joint9_formal` (all 9 languages); GPU 1 trains `new5_formal` (the 5 new languages only). Both run 2 epochs.
3. **Evaluation:**
   - base, teacher and both new models (plain and hybrid) on the new languages
   - `joint9_formal` on all nine
   - IN22-Gen forgetting check for `joint9_formal`

**Input:** only the latest bundle dataset. Use GPU T4 x2, Internet on, and the `HF_TOKEN` secret. Then **Save & Run All** (about 6 hours); the outputs end up in `/kaggle/working/outputs.zip`.

In [ ]:
# 1. Copy the newest stamped bundle into a writable working directory (+ reuse earlier drafts)
import os, shutil, subprocess, sys, glob, zipfile
from pathlib import Path

PROJECT = Path("/kaggle/working/project")
# Only folders/zips carrying kaggle/BUNDLE_VERSION count as code; outputs of earlier runs
# (added as inputs to reuse drafts) also contain a backend/ folder with OLD code.
candidates = []
for z in glob.glob("/kaggle/input/**/vocational-nmt-bundle.zip", recursive=True):
    with zipfile.ZipFile(z) as zf:
        if "kaggle/BUNDLE_VERSION" in zf.namelist():
            candidates.append((zf.read("kaggle/BUNDLE_VERSION").decode().strip(), z))
for stamp in glob.glob("/kaggle/input/**/kaggle/BUNDLE_VERSION", recursive=True):
    candidates.append((Path(stamp).read_text().strip(), str(Path(stamp).parents[1])))
if not candidates:
    raise FileNotFoundError("No bundle with kaggle/BUNDLE_VERSION under /kaggle/input - upload the latest bundle zip.")
print("bundles attached:", sorted({v for v, _ in candidates}))
version, source = max(candidates)

# /kaggle/working survives while an interactive session is open: re-copy when the attached
# bundle is newer than the copy from an earlier cell run.
copied = PROJECT / "kaggle/BUNDLE_VERSION"
if PROJECT.exists() and (not copied.exists() or copied.read_text().strip() != version):
    print("Replacing stale project copy", copied.read_text().strip() if copied.exists() else "(unstamped)")
    shutil.rmtree(PROJECT)
if not PROJECT.exists():
    print(f"Using bundle {version} from {source}")
    if source.endswith(".zip"):
        zipfile.ZipFile(source).extractall(PROJECT)
    else:
        shutil.copytree(source, PROJECT)
print("bundle version:", copied.read_text().strip())
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(PROJECT)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation during beam search
sys.path.insert(0, str(PROJECT))

CORPUS = PROJECT / "backend/data/corpus_v2"
for pattern, sub in (("drafts_*.csv", ""), ("postedit/test_*.csv", "postedit")):
    for src in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        dst = CORPUS / sub / Path(src).name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(src, dst)
            print("reused", src)
print(sorted(p.name for p in CORPUS.iterdir()))
assert (PROJECT / "backend/models/it2_lora_joint/adapter_model.safetensors").exists(), "run-4 adapter missing from bundle"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install the pinned stack (about 1 minute)
!pip install -q -r kaggle/requirements-kaggle.txt
!python -c "import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)"

In [ ]:
# 3. Hugging Face token + access check for the gated repos
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import hf_hub_download
IN22_OK = True
for repo, kind, fname in [("ai4bharat/indictrans2-en-indic-dist-200M", "model", "config.json"),
                          ("ai4bharat/indictrans2-en-indic-1B", "model", "config.json"),
                          ("ai4bharat/IN22-Gen", "dataset", "data/train-00000-of-00001.parquet")]:
    try:
        hf_hub_download(repo, fname, repo_type=kind, token=os.environ["HF_TOKEN"])
        print("access OK:", repo)
    except Exception as exc:
        print(f"NO ACCESS: {repo} -> accept its terms on huggingface.co ({type(exc).__name__})")
        if kind == "model":
            raise
        IN22_OK = False

In [ ]:
# 4. Smoke test (~6 min): Tamil + Marathi drafts and placeholder repair on 40 sentences, in a scratch corpus
def check(step):
    assert _exit_code == 0, f"{step} failed - see the output above"

import pandas as pd
SMOKE = Path("/kaggle/working/smoke_corpus")
SMOKE.mkdir(parents=True, exist_ok=True)
shutil.copy(CORPUS / "en_sentences.csv", SMOKE / "en_sentences.csv")
!LE_CORPUS_DIR=/kaggle/working/smoke_corpus CUDA_VISIBLE_DEVICES=0 python -m backend.generate_drafts --langs tam_Taml,mar_Deva --limit 120 --overwrite
check("draft smoke test")
!LE_CORPUS_DIR=/kaggle/working/smoke_corpus LE_RESULTS_DIR=/kaggle/working/smoke_results CUDA_VISIBLE_DEVICES=0 python -m backend.repair_drafts --langs tam_Taml,mar_Deva --limit 40 --overwrite
check("repair smoke test")
for s in ["ta", "mr"]:
    d = pd.read_csv(SMOKE / f"drafts_{s}_formal.csv", keep_default_na=False)
    d = d[d.term_repair != "kept"]
    print(s, d.term_repair.value_counts().to_dict())
    for _, r in d.head(4).iterrows():
        print(f"  [{r.term_repair}] {r.english_sentence[:80]}\n     {r.target_sentence[:110]}")

## Step 1: teacher drafts + terminology repair for the new languages (both GPUs)

In [ ]:
import torch
n_gpu = torch.cuda.device_count()
groups = [['mar_Deva', 'tam_Taml', 'guj_Gujr'], ['tel_Telu', 'kan_Knda']] if n_gpu >= 2 else [['mar_Deva', 'guj_Gujr', 'tam_Taml', 'tel_Telu', 'kan_Knda']]
procs = []
for gpu, langs in enumerate(groups):
    l = ",".join(langs)
    chain = f"python -m backend.generate_drafts --langs {l} && python -m backend.repair_drafts --langs {l}"
    procs.append(subprocess.Popen(["bash", "-c", chain], env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
                                  stdout=open(f"/kaggle/working/drafts_repair_gpu{gpu}.log", "w"), stderr=subprocess.STDOUT))
codes = [p.wait() for p in procs]
for gpu in range(len(groups)):
    print(open(f"/kaggle/working/drafts_repair_gpu{gpu}.log").read()[-1500:])
assert all(c == 0 for c in codes), codes
!cat backend/results/term_repair_stats_*.json
!python -m backend.analyze_terminology

In [ ]:
# Training smoke test (~3 min): 10 steps on all nine languages, then load the adapter
!CUDA_VISIBLE_DEVICES=0 python -m backend.finetune --drafts-variant formal --langs hin_Deva,gom_Deva,mai_Deva,doi_Deva,mar_Deva,guj_Gujr,tam_Taml,tel_Telu,kan_Knda --run-name smoke --max-steps 10 --eval-steps 5 --dev-per-lang 4
check("training smoke test")
!python -m backend.baseline_inference --model backend/models/it2_lora_smoke --text "Check the voltage with a voltmeter." --target-lang tam_Taml
check("adapter loading")
shutil.rmtree("backend/models/it2_lora_smoke", ignore_errors=True)

## Step 2: GPU 0 trains the 9-language model; GPU 1 trains the 5-language model, then evaluates the new languages

In [ ]:
gpu1 = None
if n_gpu >= 2:
    gpu1 = subprocess.Popen(["bash", "-c", 'python -m backend.finetune --drafts-variant formal --epochs 2 --eval-steps 1000 --dev-per-lang 100 --langs mar_Deva,guj_Gujr,tam_Taml,tel_Telu,kan_Knda --run-name new5_formal && python -m backend.run_domain_eval --langs mar_Deva,guj_Gujr,tam_Taml,tel_Telu,kan_Knda --systems base,teacher,lora_new5_formal,lora_new5_formal+hy --tag run7a'], env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"),
                            stdout=open("/kaggle/working/new5_gpu1.log", "w"), stderr=subprocess.STDOUT)

!CUDA_VISIBLE_DEVICES=0 python -m backend.finetune --drafts-variant formal --epochs 2 --eval-steps 1000 --dev-per-lang 100 --langs hin_Deva,gom_Deva,mai_Deva,doi_Deva,mar_Deva,guj_Gujr,tam_Taml,tel_Telu,kan_Knda --run-name joint9_formal
check("9-language fine-tuning")

## Step 3: evaluation

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m backend.run_domain_eval --langs hin_Deva,gom_Deva,mai_Deva,doi_Deva,mar_Deva,guj_Gujr,tam_Taml,tel_Telu,kan_Knda --systems lora_joint9_formal,lora_joint9_formal+hy --tag run7b
check("9-language evaluation")
if IN22_OK:
    !CUDA_VISIBLE_DEVICES=0 python -m backend.run_baseline_eval --skip-base --adapter backend/models/it2_lora_joint9_formal --output backend/results/in22_gen_lora_joint9.csv
if gpu1 is not None:
    print("GPU 1 exit code:", gpu1.wait())
    print(open("/kaggle/working/new5_gpu1.log").read()[-3000:])
else:
    !CUDA_VISIBLE_DEVICES=0 bash -c 'python -m backend.finetune --drafts-variant formal --epochs 2 --eval-steps 1000 --dev-per-lang 100 --langs mar_Deva,guj_Gujr,tam_Taml,tel_Telu,kan_Knda --run-name new5_formal && python -m backend.run_domain_eval --langs mar_Deva,guj_Gujr,tam_Taml,tel_Telu,kan_Knda --systems base,teacher,lora_new5_formal,lora_new5_formal+hy --tag run7a'
!python -m backend.analyze_case_endings backend/results/domain_predictions_UNVERIFIED_run7b.csv

In [ ]:
# Package everything that must come back to your machine
out = Path("/kaggle/working/outputs")
shutil.rmtree(out, ignore_errors=True)
(out / "corpus_v2" / "postedit").mkdir(parents=True)
for s in ["mr", "gu", "ta", "te", "kn"]:
    for name in [f"drafts_{s}.csv", f"drafts_{s}_formal.csv"]:
        if (CORPUS / name).exists():
            shutil.copy(CORPUS / name, out / "corpus_v2")
    if (CORPUS / "postedit" / f"test_{s}.csv").exists():
        shutil.copy(CORPUS / "postedit" / f"test_{s}.csv", out / "corpus_v2" / "postedit")
for run in ["joint9_formal", "new5_formal"]:
    src = Path(f"backend/models/it2_lora_{run}")
    if src.exists():
        shutil.copytree(src, out / src.name, ignore=shutil.ignore_patterns("checkpoints"))
    else:
        print("missing:", src)
shutil.copytree("backend/results", out / "results")
for log in glob.glob("/kaggle/working/*.log"):
    shutil.copy(log, out)
shutil.make_archive("/kaggle/working/outputs", "zip", out)
!du -sh /kaggle/working/outputs.zip